# Stepped Impedance Filter

In [7]:
# %load_ext autoreload
# %autoreload 2

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings
import pyEPR as epr
import numpy as np

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from collections import OrderedDict

### Create custom component to join CPW of different lengths

In [ ]:
from qiskit_metal.qlibrary.core import QComponent

class SteppedCPWConnector(QComponent):
    """A straight CPW connector with a step change in trace width and gap halfway through."""

    default_options = Dict(
        length='400um',  # Total length, split equally between both sections
        trace_width_1='10um',
        gap_1='6um',
        trace_width_2='20um',
        gap_2='10um',
        pos_x='0um',
        pos_y='0um',
        orientation='0',
        layer='1'
    )

    component_metadata = Dict(
        short_name='StepCPW',
        _qgeometry_table_path=True,
        _qgeometry_table_poly=True
    )

    TOOLTIP = "A CPW connector with a discrete step between two different widths and gaps."

    def make(self):
        p = self.p
        half_len = float(p.length) / 2

        # Segment 1 (left side)
        w1 = float(p.trace_width_1)
        g1 = float(p.gap_1)
        h1_total = w1 + 2*g1  # total height of CPW including gaps

        # Trace bottom-aligned at y = 0
        seg1 = draw.rectangle(half_len, w1, -half_len/2, 0)
        gap1 = draw.rectangle(half_len, w1 + 2*g1, -half_len/2, 0)
        # gap1 = gap1.difference(seg1)

        # Segment 2 (right side)
        w2 = float(p.trace_width_2)
        g2 = float(p.gap_2)
        h2_total = w2 + 2*g2

        seg2 = draw.rectangle(half_len, w2, half_len/2, 0)
        gap2 = draw.rectangle(half_len, w2 + 2*g2, half_len/2, 0)
        # gap2 = gap2.difference(seg2)

        # Union and transform
        trace = draw.union([seg1, seg2])
        gap = draw.union([gap1, gap2])

        # Apply rotation and translation
        trace = draw.rotate(trace, p.orientation, origin=(0, 0))
        gap = draw.rotate(gap, p.orientation, origin=(0, 0))
        trace = draw.translate(trace, p.pos_x, p.pos_y)
        gap = draw.translate(gap, p.pos_x, p.pos_y)

        self.add_qgeometry('poly', {'trace': trace}, layer=p.layer)
        self.add_qgeometry('poly', {'gap': gap}, subtract=True, layer=p.layer)

        # Pins: placed along the trace centerline in x, y determined from base
        vec = np.array([np.cos(np.deg2rad(p.orientation)), np.sin(np.deg2rad(p.orientation))])
        mid = np.array([float(p.pos_x), float(p.pos_y)])
        offset = vec * (float(p.length) / 2)

        start_pt = mid - offset
        end_pt = mid + offset

        # Pins placed at the ends of the CPW trace
        cpw_left = draw.LineString([start_pt, start_pt - vec * 1e-6])
        cpw_right = draw.LineString([end_pt, end_pt + vec * 1e-6])

        self.add_pin('pin1', cpw_left.coords, width=w1, gap=g1, input_as_norm=True)
        self.add_pin('pin2', cpw_right.coords, width=w2, gap=g2, input_as_norm=True)


In [9]:
design = designs.DesignPlanar({}, True)

# Set up chip dimensions 
design.chips.main.size.size_x = '5mm'
design.chips.main.size.size_y = '5mm'
design.chips.main.size.size_z = '-250um' # if using the two-inch wafers
# Resonator and feedline gap width (W) and center conductor width (S) from reference 2
design.variables['cpw_width'] = '15.5 um' #S from reference 2
design.variables['cpw_gap'] = '9 um' #W from reference 2
design.chips.main.size.sample_holder_top='1400um'
design.chips.main.size.sample_holder_bottom='250um'

design.overwrite_enabled = True
hfss = design.renderers.hfss

# Open GUI
gui = MetalGUI(design)

### Define Parameters

In [10]:
w_Zlo = '166um'
g_Zlo = '7um'

w_Zhi = '10um'
g_Zhi = '85um'

# trying to keep w+2*g same
# print(7*2 + 166, 85*2 + 10)

ref_X = '-1.5mm'
ref_Y = '0mm'

In [11]:
import numpy as np
from scipy.special import ellipk
from IPython.display import display, Markdown

column_width = 15

# material properties
mu0 = 4*np.pi*1e-7
ep0 = 8.854187817e-12
er = 11.7 # Relative permittivity of silicon
e_eff = (1 + er)/2

# cpw geometry
s = 10e-6 # width of the center conductor
w = 85e-6 # width of the gap
l = 4000e-6 # length of the CPW

k = s/(s+2*w) # cpw fill factor
kp = np.sqrt(1-k**2) # elliptic modulus, is it really called this??

# Evaluate the elliptic integral of the first kind
K = ellipk(k)
Kp = ellipk(kp)

Lksq = 0.6e-12  # kinetic inductance per square
Lk = Lksq/s # kinetic inductance per unit length

Ll = mu0* Kp/(4*K)
Cl = 4*ep0*e_eff* K/Kp

Lt = Ll + Lk  # Total inductance per unit length

vph = 1/np.sqrt(Lt*Cl)
Zc = np.sqrt(Lt/Cl)
f0 = 1/(4*l*np.sqrt(Lt*Cl))  # Fundamental frequency

# Create the table
header = f"| {'Parameter':<{column_width}} | {'Value (S.I.)':<{column_width}} |\n"
header += f"|{'-' * (column_width + 2)}|{'-' * (column_width + 2)}|\n"
values = (
    f"| $s$ | {s:.5e} |\n"
    f"| $w$ | {w:.5e} |\n"
    f"| $L_l$ | {Ll:.5e} |\n"
    f"| $C_l$ | {Cl:.5e} |\n"
    f"| $L_k$ | {Lk:.5e} |\n"
    f"| $L_t$ | {Lt:.5e} |\n"
    f"| $Zc$ | {Zc:.5e} |\n"
    f"| $v_{{ph}}$ | {vph:.5e} |\n"
    f"| $f_0$ | {f0:.5e} |\n"
    f"| $\\lambda_g/4$ | {l:.5e} |\n"
)

# Combine header and values
table = header + values

# Display the table in Markdown
display(Markdown(table))

| Parameter       | Value (S.I.)    |
|-----------------|-----------------|
| $s$ | 1.00000e-05 |
| $w$ | 8.50000e-05 |
| $L_l$ | 9.11775e-07 |
| $C_l$ | 7.74898e-11 |
| $L_k$ | 6.00000e-08 |
| $L_t$ | 9.71775e-07 |
| $Zc$ | 1.11985e+02 |
| $v_{ph}$ | 1.15238e+08 |
| $f_0$ | 7.20236e+09 |
| $\lambda_g/4$ | 4.00000e-03 |


#### Concise and automated

In [12]:
def CreatePathfinder(design, name, start_comp, start_pin, end_comp, end_pin, anchors=None, options=None):
    """Helper function to create a RoutePathfinder with common options."""
    return RoutePathfinder(design, name, options=Dict(
        pin_inputs=Dict(
            start_pin=Dict(component=start_comp, pin=start_pin),
            end_pin=Dict(component=end_comp, pin=end_pin)),
        anchors=anchors,
        **(options or {})
    ))

def CreateSteppedConnector(design, name, pos_x, pos_y, orientation, options):
    """Helper function to create a SteppedCPWConnector with common options."""
    return SteppedCPWConnector(design, name, options=Dict(
        pos_x=pos_x,
        pos_y=pos_y,
        orientation=orientation,
        **options
    ))

optionsZhi = Dict(total_length='4mm', hfss_wire_bonds=True, 
                  lead=Dict(start_straight='0.05mm', end_straight='0.05mm'),
                  trace_width=w_Zhi,trace_gap=g_Zhi, fillet='200um',)

optionsZlo = Dict(total_length='4mm', hfss_wire_bonds=True,
                  lead=Dict(start_straight='0.1mm', end_straight='0.1mm'),
                  trace_width=w_Zlo, trace_gap=g_Zlo,fillet='200um')

optionsOddStep = Dict(length='50um', trace_width_1=w_Zlo, gap_1=g_Zlo, trace_width_2=w_Zhi, gap_2=g_Zhi)
optionsEvenStep = Dict(length='50um', trace_width_1=w_Zhi, gap_1=g_Zhi, trace_width_2=w_Zlo, gap_2=g_Zlo)

LP1 = LaunchpadWirebondDriven(design,name='LP1',
                       options={'orientation': '90', 'pos_x': '-1.5mm', 'pos_y': '200um','pad_width':w_Zlo,'pad_height':w_Zlo,'pad_gap':'90um', 
                                'trace_width':w_Zlo , 'trace_gap': g_Zlo, 'taper_height': '0.12mm'},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

LP2 = LaunchpadWirebondDriven(design,name='LP2',
                       options={'orientation': '-90', 'pos_x': '0.5mm', 'pos_y': '0mm','pad_width':'180um','pad_height':'180um','pad_gap':'140um',
                                'trace_width':w_Zhi, 'trace_gap': g_Zhi},
                       component_template={'falseparam1': {'falseparam2': 'false-param', 'falseparam3': 'false-param'}},
                                        )

step1 = CreateSteppedConnector(design, 'step1', '-1mm', '0mm', '-90', optionsOddStep)
SegLow1 = CreatePathfinder(design, 'SegLow1', 'LP1', 'tie', 'step1', 'pin1', OrderedDict({0: np.array([-1.5, 2])}), optionsZlo)

step2 = CreateSteppedConnector(design, 'step2', '-0.5mm', '0mm', '90', optionsEvenStep)
SegHigh1 = CreatePathfinder(design, 'SegHigh1', 'step1', 'pin2', 'step2', 'pin1', OrderedDict({0: np.array([-0.5, -2.0])}), optionsZhi)

step3 = CreateSteppedConnector(design, 'step3', '0mm', '0mm', '-90', optionsOddStep)
SegLow2 = CreatePathfinder(design, 'SegLow2', 'step2', 'pin2', 'step3', 'pin1', OrderedDict({0: np.array([0, 2])}), optionsZlo)

# OTG3 = OpenToGround(design, name="OTG3", 
#     options={
#         "pos_x": "0.5mm",               # X position
#         "pos_y": "0mm",              # Y position
#         "orientation": "90",           # Orientation (0, 90, 180, or 270 degrees)
#         "layer": '1'                   # Specify the layer if needed
#     }
# )

# SegHigh2 = CreatePathfinder(design, 'SegHigh2', 'step3', 'pin2', 'OTG3', 'open', OrderedDict({0: np.array([0.5, -2])}), optionsZhi)
SegHigh2 = CreatePathfinder(design, 'SegHigh2', 'step3', 'pin2', 'LP2', 'tie', OrderedDict({0: np.array([0.5, -2])}), optionsZhi)

gui.rebuild()  # Rebuild the design with the new components 
gui.autoscale()

In [ ]:
segment_anchors = 
{'SegHigh2': OrderedDict([(0, np.array([ 0.5      , -1.8603834]))]),
 'SegHigh1': OrderedDict([(0, np.array([-0.5      , -1.8603834]))]),
 'SegLow1': OrderedDict([(0, np.array([-1.5       ,  1.81131066]))]),
 'SegLow2': OrderedDict([(0, np.array([0.        , 1.86059219]))])}

In [200]:
def auto_adjust_anchor(design, name, start_comp, start_pin, end_comp, end_pin, 
                       anchor_x, anchor_y_start, 
                       options=None, target_length_mm=4.0, tolerance=0.001,
                       anchor_dict=None):
    """
    Adjusts the anchor point to match the target length within specified tolerance.
    Uses adaptive step size (delta). Updates anchor_dict[name] with final anchor.
    """
    max_iter = 30
    y = anchor_y_start
    delta = 0.2  # initial step size (mm)

    target_length_mm = float(str(target_length_mm).strip('mm'))
    
    direction = 1
    prev_error = 0

    for i in range(max_iter):

        anchor = OrderedDict({0: np.array([anchor_x, y])})
        seg = CreatePathfinder(design, name, start_comp, start_pin, end_comp, end_pin, anchor, options)
        design.rebuild()
        actual_length = float(seg.length)
        error = abs(actual_length - target_length_mm)

        print(i, y, actual_length, error, prev_error, direction)

        if error < tolerance:
            print(f"[✓] Match found: y = {y:.4f} mm → Length = {actual_length:.4f} mm")
            if anchor_dict is not None:
                anchor_dict[name] = anchor  # update global or shared anchor store
            return seg

        direction = -np.sign(error - prev_error)*direction

        # Adaptive step size
        delta = delta * 0.5 if error > prev_error else delta * 1.05

        y += direction * delta
        prev_error = error

    raise RuntimeError("❌ Failed to converge to the target length within max iterations.")


SegHigh2 = auto_adjust_anchor(
    design=design,
    name='SegHigh2',
    start_comp='step3',
    start_pin='pin2',
    end_comp='LP2',
    end_pin='tie',
    anchor_x= segment_anchors['SegHigh2'][0][0],
    anchor_y_start = segment_anchors['SegHigh2'][0][1],
    options=optionsZhi,
    target_length_mm=optionsZhi['total_length'],
    tolerance=0.001,
    anchor_dict=segment_anchors,
)

SegHigh1 = auto_adjust_anchor(
    design=design,
    name='SegHigh1',
    start_comp='step1',
    start_pin='pin2',
    end_comp='step2',
    end_pin='pin1',
    anchor_x= segment_anchors['SegHigh1'][0][0],
    anchor_y_start= segment_anchors['SegHigh1'][0][1],
    options=optionsZhi,
    target_length_mm=optionsZhi['total_length'],
    tolerance=0.001,
    anchor_dict=segment_anchors,
)

SegLow1 = auto_adjust_anchor(
    design=design,
    name='SegLow1',
    start_comp='LP1',
    start_pin='tie',
    end_comp='step1',
    end_pin='pin1',
    anchor_x= segment_anchors['SegLow1'][0][0],
    anchor_y_start= segment_anchors['SegLow1'][0][1],
    options=optionsZlo,
    target_length_mm=optionsZlo['total_length'],
    tolerance=0.001,
    anchor_dict=segment_anchors,
)

SegLow2 = auto_adjust_anchor(
    design=design,
    name='SegLow2',
    start_comp='step2',
    start_pin='pin2',
    end_comp='step3',
    end_pin='pin1',
    anchor_x=segment_anchors['SegLow2'][0][0],
    anchor_y_start=segment_anchors['SegLow2'][0][1],
    options=optionsZlo,
    target_length_mm=optionsZlo['total_length'],
    tolerance=0.001,
    anchor_dict=segment_anchors,
)

gui.rebuild()
gui.autoscale()

0 -1.8603833977105402 3.999084326139039 0.0009156738609608084 0 1
[✓] Match found: y = -1.8604 mm → Length = 3.9991 mm
0 -1.8603833977105402 3.999083326139039 0.0009166738609609482 0 1
[✓] Match found: y = -1.8604 mm → Length = 3.9991 mm
0 1.8113106640406835 4.0009388587993255 0.0009388587993255371 0 1
[✓] Match found: y = 1.8113 mm → Length = 4.0009 mm
0 1.8605921875000002 3.9995009057179596 0.0004990942820404243 0 1
[✓] Match found: y = 1.8606 mm → Length = 3.9995 mm


## Rendering to Ansys

In [154]:
from qiskit_metal.analyses.simulation.scattering_impedance import ScatteringImpedanceSim
# em1 = ScatteringImpedanceSim(design, "hfss")

# hfss = em1.renderer
# hfss.start()

from qiskit_metal.analyses.quantization import EPRanalysis
import pyEPR as epr
eig_qres = EPRanalysis(design, "hfss")
hfss = eig_qres.sim.renderer
hfss.start()

INFO 04:20PM [connect_project]: Connecting to Ansys Desktop API...
INFO 04:20PM [load_ansys_project]: 	Opened Ansys App
INFO 04:20PM [load_ansys_project]: 	Opened Ansys Desktop v2024.1.0
INFO 04:20PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/AMS715/AnsysHFSSProjects/
	Project:   Project1
INFO 04:20PM [connect_design]: No active design found (or error getting active design).
INFO 04:20PM [connect]: 	 Connected to project "Project1". No design detected


True

In [155]:
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '5mm',
  'size_y': '5mm',
  'size_z': '-250um',
  'sample_holder_top': '1400um',
  'sample_holder_bottom': '250um'}}

In [156]:
# hfss.clean_active_design()
# hfss.activate_ansys_design("HangingResonatorsSparam", 'drivenmodal')
hfss.activate_ansys_design("SIF", 'eigenmode')

04:21PM 23s WARNING [activate_ansys_design]: The design_name=SIF was not in active project.  Designs in active project are: 
[].  A new design will be added to the project.  
INFO 04:21PM [connect_design]: 	Opened active design
	Design:    SIF [Solution type: Eigenmode]
WARNING 04:21PM [connect_setup]: 	No design setup detected.
WARNING 04:21PM [connect_setup]: 	Creating eigenmode default setup.
INFO 04:21PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


In [157]:
design._chips.main.size

{'center_x': '0.0mm',
 'center_y': '0.0mm',
 'center_z': '0.0mm',
 'size_x': '5mm',
 'size_y': '5mm',
 'size_z': '-250um',
 'sample_holder_top': '1400um',
 'sample_holder_bottom': '250um'}

In [158]:
hfss.render_design(selection=[],
                   open_pins=[],
                #    port_list=[('cpw_openRight', 'end', 50), ('cpw_openLeft', 'end', 50)],
                   port_list=[('LP1', 'in','50'), ('LP2', 'in','50')],
                   jj_to_port=[],
                   ignored_jjs=[('Q1', 'rect_jj'), ('Q2', 'rect_jj')],
                   box_plus_buffer = False)

In [ ]:
hfss.add_sweep(setup_name="Setup1",
               name="Sweep",
               start_ghz=4.0,
               stop_ghz=8.0,
               count=2001,
               type="Interpolating")

OSError: Setup Setup1 not found: ('Setup',)

: 